In [14]:
import os
from typing import List

from dotenv import load_dotenv

from langchain_community.document_loaders import (
    PyPDFLoader,
    DirectoryLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

c:\Users\Rhythm Ahsan\anaconda3\envs\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found in .env")

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("Environment variables loaded successfully")

Environment variables loaded successfully


In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()
    return documents


extracted_data = load_pdf_files("data")

print("Number of documents:", len(extracted_data))

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNADH+Berkeley-BookItalic', '/FirstChar': 32, '/LastChar': 111, '/Subtype': '/Type1', '/FontDescriptor': IndirectObject(8788, 0, 1567126012320), '/Widths': [241, 241, 352, 481, 481, 667, 778, 204, 315, 315, 500, 481, 241, 333, 241, 278, 481, 481, 481, 481, 481, 481, 481, 481, 481, 481, 241, 241, 481, 481, 481, 352, 800, 593, 519, 611, 685, 537, 519, 667, 685, 296, 296, 611, 500, 796, 685, 685, 519, 667, 556, 463, 556, 685, 593, 870, 593, 519, 556, 315, 278, 315, 481, 500, 241, 463, 444, 370, 463, 389, 259, 407, 463, 241, 241, 444, 241, 722, 481, 426]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNAFK+Berkeley-Bold', '/FirstChar': 32,

Number of documents: 637


In [10]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs = []

    for doc in docs:
        src = doc.metadata.get("source")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs


minimal_docs = filter_to_minimal_docs(extracted_data)

print("Minimal documents:", len(minimal_docs))

Minimal documents: 637


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    texts_chunk = text_splitter.split_documents(minimal_docs)

    return texts_chunk


texts_chunk = text_split(minimal_docs)

print("Number of chunks:", len(texts_chunk))

Number of chunks: 5860


In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )

    return embeddings


embedding = download_embeddings()

vector = embedding.embed_query("Hello world")

print("Embedding dimension:", len(vector))

C:\Users\Rhythm Ahsan\AppData\Local\Temp\ipykernel_28140\701338115.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\Rhythm Ahsan\anaconda3\envs\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2293.51it/s]


Embedding dimension: 384


In [16]:
from pinecone import Pinecone, ServerlessSpec

# Make sure your API key is available
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is not loaded")

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=len(vector),
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("Pinecone index ready")

Pinecone index ready


In [19]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

print("Documents uploaded to Pinecone successfully")

Documents uploaded to Pinecone successfully


In [20]:
retriever = docsearch.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever ready")

Retriever ready


In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash"
)

response = chatModel.invoke("Hello, are you working?")

print(response.content)

Hello! Yes, I am working and ready to help. How can I assist you today?


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are a helpful medical assistant.

You will be provided with context from medical documents.

Answer the user's question based only on the provided context.

If the answer is not present in the context, say:
"I don't know based on the provided medical documents."

Do not make up medical information.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

print("Prompt created successfully")

Prompt created successfully


In [26]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

question_answer_chain = create_stuff_documents_chain(
    chatModel,
    prompt
)

rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

print("RAG chain created successfully")

RAG chain created successfully


In [27]:
response = rag_chain.invoke({
    "input": "What is the treatment for diabetes?"
})

print(response["answer"])

Based on the provided documents, antidiabetic drugs are medicines used to help control blood sugar levels in people with diabetes mellitus (sugar diabetes).


In [28]:
response = rag_chain.invoke({
    "input": "What is acne?"
})

print(response["answer"])

Based on the provided medical documents, acne is the general name given to a skin disorder in which the sebaceous glands become inflamed.


In [29]:
response = rag_chain.invoke({
    "input": "What is the treatment for acne?"
})

print(response["answer"])

Based on the provided medical documents, the treatment for acne depends on whether it is mild, moderate, or severe:

* **Mild noninflammatory acne:** Treated with topical drugs to reduce the formation of new comedones. These include:
  * Topical tretinoin (which increases the turnover of skin cells)
  * Benzoyl peroxide
  * Adapalene
  * Salicylic acid
* **When complicated by inflammation:** Topical antibiotics may be added to the treatment regimen. 

Improvement with these treatments is usually seen in two to four weeks.
